# Consolidação das Avaliações — Método 1

Junta os 12 arquivos de avaliação (3 editais × 2 ferramentas × 2 avaliadores-LLM) em um único DataFrame no formato de `df_avaliacoes.xlsx`.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Mapeamento dos arquivos

Cada arquivo de avaliação contém as colunas `id`, `pergunta`, `resposta` e as avaliações de **um** LLM avaliador.  
Os pares `(gpt_eval, opus_eval)` são mergeados por `id` para formar uma linha por pergunta.

Ajuste o `DATA_DIR` e os metadados (`edital`, `modelo`, `provedor`) conforme necessário.

In [7]:
DATA_DIR = Path(".")  # pasta onde estão os xlsx de avaliação

AVALIACOES = [
    # ── BNDES ──────────────────────────────────────────────────────────────────
    ("bndes", "std_chatgpt", "openai",    "bndes_std_chatgpt_gpt_eval.xlsx",  "bndes_std_chatgpt_opus_eval.xlsx"),
    ("bndes", "std_claude",  "anthropic", "bndes_std_claude_gpt_eval.xlsx",   "bndes_std_claude_opus_eval.xlsx"),
    # ── CVM ────────────────────────────────────────────────────────────────────
    ("cvm",   "std_chatgpt", "openai",    "cvm_std_chatgpt_gpt_eval.xlsx",    "cvm_std_chatgpt_opus_eval.xlsx"),
    ("cvm",   "std_claude",  "anthropic", "cvm_std_claude_gpt_eval.xlsx",     "cvm_std_claude_opus_eval.xlsx"),
    # ── PETROBRAS ──────────────────────────────────────────────────────────────
    ("petrobras", "std_chatgpt",    "openai",    "petrobras_std_chatgpt_gpt_eval.xlsx",    "petrobras_std_chatgpt_opus_eval.xlsx"),
    ("petrobras", "std_claude", "anthropic", "petrobras_std_claude_gpt_eval.xlsx", "petrobras_std_claude_opus_eval.xlsx"),
]

print(f"{len(AVALIACOES)} pares de avaliação mapeados.")

6 pares de avaliação mapeados.


In [8]:
# id → categoria (fixo para todos os editais)
CATEGORIA = {
    1:  "Dados sobre as inscrições",
    2:  "Dados sobre as inscrições",
    3:  "Dados sobre as inscrições",
    4:  "Dados sobre as inscrições",
    5:  "Dados sobre as inscrições",
    6:  "Dados sobre as inscrições",
    7:  "Dados sobre as inscrições",
    8:  "Dados sobre as inscrições",
    9:  "Dados sobre as inscrições",
    10: "Dados sobre as inscrições",
    11: "Dados sobre o Concurso",
    12: "Dados sobre o Concurso",
    13: "Dados sobre o Concurso",
    14: "Dados sobre o Concurso",
    15: "Dados sobre o Concurso",
    16: "Dados sobre o Concurso",
    17: "Dados sobre o Concurso",
    18: "Dados sobre o Cargo",
    19: "Dados sobre o Cargo",
    20: "Dados sobre o Cargo",
    21: "Dados sobre o Cargo",
    22: "Dados sobre o Cargo",
    23: "Dados sobre o Cargo",
    24: "Dados sobre o Cargo",
    25: "Dados sobre o Cargo",
    26: "Dados sobre o Cargo",
    27: "Dados sobre o Cargo",
    28: "Dados sobre o Cargo",
    29: "Dados sobre o Cargo",
    30: "Dados sobre o Cargo",
    31: "Dados sobre a Prova",
    32: "Dados sobre a Prova",
    33: "Dados sobre a Prova",
    34: "Dados sobre a Prova",
    35: "Dados sobre a Prova",
    36: "Dados sobre a Prova",
    37: "Dados sobre a Prova",
    38: "Dados sobre a Prova",
    39: "Dados sobre a Prova",
    40: "Dados sobre a Prova",
    41: "Dados sobre a Prova",
    42: "Dados sobre a Prova",
    43: "Dados sobre a Prova",
    44: "Dados sobre a Prova",
    45: "Dados sobre a Prova",
    46: "Dados sobre Procedimentos",
    47: "Dados sobre Procedimentos",
    48: "Dados sobre Procedimentos",
    49: "Dados sobre Procedimentos",
    50: "Dados sobre Procedimentos",
}

## 3. Função de consolidação

In [9]:
def consolidar_par(
    edital: str,
    modelo: str,
    provedor: str,
    arquivo_gpt: str,
    arquivo_opus: str,
    data_dir: Path = DATA_DIR,
) -> pd.DataFrame:
    """Lê um par de arquivos (gpt_eval + opus_eval) e retorna um DataFrame
    no formato de df_avaliacoes.xlsx."""

    df_gpt = pd.read_excel(data_dir / arquivo_gpt)
    df_opus = pd.read_excel(data_dir / arquivo_opus)

    # Colunas esperadas
    assert {"id", "pergunta", "resposta", "justificativa_gpt", "avaliacao_gpt"}.issubset(df_gpt.columns), \
        f"{arquivo_gpt} precisa ter: id, pergunta, resposta, justificativa_gpt, avaliacao_gpt"
    assert {"id", "justificativa_opus", "avaliacao_opus"}.issubset(df_opus.columns), \
        f"{arquivo_opus} precisa ter: id, justificativa_opus, avaliacao_opus"

    # Merge por id
    df = df_gpt[["id", "pergunta", "resposta", "avaliacao_gpt", "justificativa_gpt"]].merge(
        df_opus[["id", "avaliacao_opus", "justificativa_opus"]],
        on="id",
        how="outer",
        validate="1:1",
    )

    # Metadados
    df.insert(0, "edital", edital)
    df.insert(1, "modelo", modelo)
    df.insert(2, "provedor", provedor)

    df["categoria"] = df["id"].map(CATEGORIA)

    # Convergência: True quando as duas notas são iguais
    df["convergencia"] = df["avaliacao_gpt"] == df["avaliacao_opus"]

    # Ordem final
    colunas = [
        "edital", "modelo", "provedor", "id", "categoria",
        "pergunta", "resposta",
        "avaliacao_gpt", "justificativa_gpt",
        "avaliacao_opus", "justificativa_opus",
        "convergencia",
    ]
    return df[colunas]

## 4. Processar todos os pares e consolidar

In [10]:
frames = []

for edital, modelo, provedor, arq_gpt, arq_opus in AVALIACOES:
    gpt_path  = DATA_DIR / arq_gpt
    opus_path = DATA_DIR / arq_opus

    if not gpt_path.exists():
        print(f"[AVISO] Arquivo não encontrado: {arq_gpt}")
        continue
    if not opus_path.exists():
        print(f"[AVISO] Arquivo não encontrado: {arq_opus}")
        continue

    df_par = consolidar_par(edital, modelo, provedor, arq_gpt, arq_opus)
    frames.append(df_par)
    print(f"✓ {edital} | {modelo} ({provedor}) — {len(df_par)} linhas")

df_consolidado = pd.concat(frames, ignore_index=True)
print(f"\nTotal consolidado: {len(df_consolidado)} linhas")

✓ bndes | std_chatgpt (openai) — 50 linhas
✓ bndes | std_claude (anthropic) — 50 linhas
✓ cvm | std_chatgpt (openai) — 50 linhas
✓ cvm | std_claude (anthropic) — 50 linhas
✓ petrobras | std_chatgpt (openai) — 50 linhas
✓ petrobras | std_claude (anthropic) — 50 linhas

Total consolidado: 300 linhas


## 5. Validação rápida

In [11]:
print("Shape:", df_consolidado.shape)
print("\nNulos por coluna:")
print(df_consolidado.isnull().sum())
print("\nDistribuição por edital / modelo:")
print(df_consolidado.groupby(["edital", "modelo"])["id"].count().to_string())
print("\nConvergência geral:")
print(df_consolidado["convergencia"].value_counts())

Shape: (300, 12)

Nulos por coluna:
edital                0
modelo                0
provedor              0
id                    0
categoria             0
pergunta              0
resposta              0
avaliacao_gpt         0
justificativa_gpt     0
avaliacao_opus        0
justificativa_opus    0
convergencia          0
dtype: int64

Distribuição por edital / modelo:
edital     modelo     
bndes      std_chatgpt    50
           std_claude     50
cvm        std_chatgpt    50
           std_claude     50
petrobras  std_chatgpt    50
           std_claude     50

Convergência geral:
convergencia
True     271
False     29
Name: count, dtype: int64


In [12]:
df_consolidado.head(3)

,edital,modelo,provedor,id,categoria,pergunta,resposta,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia
0,bndes,std_chatgpt,openai,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,O período de inscrição para a Seleção Pública ...,1.0,Certo. O item 6.2.1 (p. 9) prevê inscrição ent...,1.0,Subitens 6.2.1 e 6.3.1 do edital: inscrições n...,True
1,bndes,std_chatgpt,openai,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,O valor da taxa de inscrição do concurso do\nB...,1.0,Certo. O item 6.2.3 (p. 9) fixa o recolhimento...,1.0,Subitem 6.2.3 do edital: 'O recolhimento do va...,True
2,bndes,std_chatgpt,openai,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,A inscrição do concurso do\nBanco Nacional de ...,1.0,Certo. Os itens 6.2.1 e 6.3.1 (p. 9-10) indica...,1.0,Subitem 6.2.1 do edital: inscrição na página d...,True


## 6. Salvar resultado

In [13]:
OUTPUT = DATA_DIR / "analise_out" / "analise_consolidada_std_chats.xlsx"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

df_consolidado.to_excel(OUTPUT, index=False)
print(f"Salvo em: {OUTPUT}")

Salvo em: analise_out/analise_consolidada_std_chats.xlsx


In [22]:
df_consolidado[df_consolidado["convergencia"] == False].sort_values(["edital", "id"]).to_excel('chats_div.xlsx')

In [20]:
resumo = (
    df_consolidado
    .groupby(["edital", "modelo"])
    .agg(
        nota_abs_gpt=("avaliacao_gpt", "sum"),
        pct_gpt=("avaliacao_gpt", "mean"),
        nota_abs_opus=("avaliacao_opus", "sum"),
        pct_opus=("avaliacao_opus", "mean"),
    )
    .reset_index()
)

resumo["pct_gpt"] = resumo["pct_gpt"].map("{:.1%}".format)
resumo["pct_opus"] = resumo["pct_opus"].map("{:.1%}".format)

resumo

,edital,modelo,nota_abs_gpt,pct_gpt,nota_abs_opus,pct_opus
0,bndes,std_chatgpt,38.0,76.0%,39.0,78.0%
1,bndes,std_claude,41.5,83.0%,49.5,99.0%
2,cvm,std_chatgpt,48.5,97.0%,50.0,100.0%
3,cvm,std_claude,49.5,99.0%,50.0,100.0%
4,petrobras,std_chatgpt,46.0,92.0%,46.0,92.0%
5,petrobras,std_claude,49.5,99.0%,50.0,100.0%


In [21]:
resumo_modelo = (
    df_consolidado
    .groupby("modelo")
    .agg(
        nota_abs_gpt=("avaliacao_gpt", "sum"),
        pct_gpt=("avaliacao_gpt", "mean"),
        nota_abs_opus=("avaliacao_opus", "sum"),
        pct_opus=("avaliacao_opus", "mean"),
    )
    .reset_index()
)

resumo_modelo["pct_gpt"] = resumo_modelo["pct_gpt"].map("{:.1%}".format)
resumo_modelo["pct_opus"] = resumo_modelo["pct_opus"].map("{:.1%}".format)

resumo_modelo

,modelo,nota_abs_gpt,pct_gpt,nota_abs_opus,pct_opus
0,std_chatgpt,132.5,88.3%,135.0,90.0%
1,std_claude,140.5,93.7%,149.5,99.7%
